In [ ]:
# Поместите pdf которые хотите добавить в папку hypothesis-factory\data\raw_to_add после чего запустите следующую ячейку

In [1]:
# ==========================================================
# УТИЛИТА: ДОБАВЛЕНИЕ НОВЫХ PDF В ПОСТОЯННУЮ БАЗУ ДАННЫХ
# ==========================================================
import os
import sys
import glob
import time
import torch
import chromadb
from sentence_transformers import SentenceTransformer

project_root = os.path.abspath('..')
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

from hypothesis_factory.ingestion_gpu import GpuDoclingReader

# Настройка путей
chroma_db_dir = os.path.join(project_root, "data", "chroma_db")
raw_data_dir = os.path.join(project_root, "data", "raw_to_add") # Отдельная папка для новых файлов!

# Создадим папку для новых файлов, если её нет
os.makedirs(raw_data_dir, exist_ok=True)

# 1. Проверяем наличие файлов для добавления
pdf_files = glob.glob(os.path.join(raw_data_dir, "*.pdf"))
if not pdf_files:
    print(f" Папка '{raw_data_dir}' пуста.")
    print("Положите туда новые PDF файлы и запустите ячейку снова.")
    sys.exit()

print(f"Найдено {len(pdf_files)} новых файлов для добавления в базу.")

# 2. Инициализируем GPU и модели
cuda_ok = torch.cuda.is_available()
DEVICE = "cuda" if cuda_ok else "cpu"
print(f"Загрузка модели эмбеддингов на {DEVICE}...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large', device=DEVICE)

# 3. Подключаемся к существующей базе данных (или создаем, если первый раз)
print("\nПодключение к ChromaDB...")
persistent_client = chromadb.PersistentClient(path=chroma_db_dir)
collection = persistent_client.get_or_create_collection("materials_db_notebook")
initial_count = collection.count()
print(f"Текущее количество чанков в базе: {initial_count}")

# 4. Настраиваем парсер (Docling)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.chunking import HierarchicalChunker

accelerator_options = AcceleratorOptions(
    num_threads=6, 
    device=AcceleratorDevice.CUDA if cuda_ok else AcceleratorDevice.CPU,
)
pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options = accelerator_options
pipeline_options.do_ocr = False              
pipeline_options.do_table_structure = False  

_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

reader = GpuDoclingReader()
reader.converter = _converter
reader.chunker = HierarchicalChunker()

# 5. Парсинг и добавление
documents, metadatas, ids = [], [], []

print("\nНачинаем парсинг...")
for pdf_path in pdf_files:
    fname = os.path.basename(pdf_path)
    t0 = time.time()
    
    try:
        chunks = reader.read(pdf_path)
        dt = time.time() - t0
        
        # Генерируем уникальный префикс для ID чанков этого файла, 
        # чтобы избежать коллизий с уже существующими в базе
        file_hash = str(hash(fname))[-6:] 
        
        for chunk in chunks:
            documents.append(chunk.text)
            # Делаем ID уникальными: старый_id + хэш_файла
            ids.append(f"{chunk.chunk_id}_{file_hash}")
            metadatas.append({
                "source_id": fname, # Записываем имя файла для цитирования
                "title": chunk.metadata.title or fname,
                "source_type": chunk.metadata.source_type
            })
            
        print(f"  {fname}: {dt:.1f} сек, извлечено {len(chunks)} чанков")
        
        # Опционально: переместить файл после успешного парсинга, чтобы не парсить дважды
        # os.rename(pdf_path, os.path.join(project_root, "data", "processed", fname))
        
    except Exception as e:
        print(f" Ошибка при парсинге {fname}: {str(e)}")

if not documents:
    print("Не удалось извлечь текст ни из одного файла.")
    sys.exit()

# 6. Векторизация и сохранение
print(f"\nНачинаем векторизацию {len(documents)} новых фрагментов...")
t0 = time.time()
embeddings = embed_model.encode(
    documents,
    batch_size=64 if cuda_ok else 16,
    show_progress_bar=True,
).tolist()
print(f"Векторизация заняла {time.time() - t0:.1f} сек")

print("Запись в базу данных...")
collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

final_count = collection.count()
print("\n" + "=" * 60)
print(" ГОТОВО! БАЗА ДАННЫХ ОБНОВЛЕНА")
print("=" * 60)
print(f"Было чанков:  {initial_count}")
print(f"Добавлено:    {len(documents)}")
print(f"Стало чанков: {final_count}")

Найдено 5 новых файлов для добавления в базу.
Загрузка модели эмбеддингов на cuda...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Подключение к ChromaDB...
Текущее количество чанков в базе: 2358

Начинаем парсинг...


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  geokniga-tehnologiyaobogashcheniyapoleznyhiskopaemyh.pdf: 6.7 сек, извлечено 238 чанков
  geokniga-flotacionnye-metody-obogashcheniya_0.pdf: 45.1 сек, извлечено 2389 чанков
  tehnologiya_izvlecheniya_zolota_i_serebra_iz_upornogo_zolotosoderzhaschego.pdf: 1.2 сек, извлечено 91 чанков
  geokniga_lodeyshchikovvvtehnologiyaizvlecheniyazolotaiserebraizupornyh1.pdf: 32.0 сек, извлечено 0 чанков
  geokniga-metallurgiya-blagorodnyh-metallov_0.pdf: 31.7 сек, извлечено 2358 чанков

Начинаем векторизацию 5076 новых фрагментов...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Векторизация заняла 66.5 сек
Запись в базу данных...

 ГОТОВО! БАЗА ДАННЫХ ОБНОВЛЕНА
Было чанков:  2358
Добавлено:    5076
Стало чанков: 7434
